# Legal RAG — Evaluation Analysis

This notebook loads `evaluation_results.json` and produces all the charts and tables you need for your README and portfolio.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Load results
with open('../evaluation_results.json') as f:
    data = json.load(f)

summary  = data['summary']
ablation = data.get('ablation', {})
per_query = pd.DataFrame(data['per_query'])

print('Summary metrics:')
for k, v in summary.items():
    print(f'  {k}: {v:.3f}')

## 1. Faithfulness Score Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
ax = axes[0]
scores = per_query['faithfulness']
colors = ['#E24B4A' if s < 0.5 else '#BA7517' if s < 0.7 else '#1D9E75' for s in scores]
ax.hist(scores, bins=20, color='#378ADD', edgecolor='white', linewidth=0.5)
ax.axvline(scores.mean(), color='#E24B4A', linestyle='--', linewidth=1.5, label=f'Mean: {scores.mean():.2f}')
ax.axvline(0.7, color='#BA7517', linestyle=':', linewidth=1.5, label='Warning threshold (0.70)')
ax.set_xlabel('Faithfulness Score')
ax.set_ylabel('Query Count')
ax.set_title('Faithfulness Score Distribution')
ax.legend()
ax.spines[['top','right']].set_visible(False)

# By doc type
ax2 = axes[1]
if 'doc_type' in per_query.columns:
    type_means = per_query.groupby('doc_type')['faithfulness'].mean().sort_values()
    palette = ['#378ADD', '#1D9E75', '#BA7517', '#D85A30']
    bars = ax2.barh(type_means.index, type_means.values,
                    color=palette[:len(type_means)], edgecolor='white')
    for bar, val in zip(bars, type_means.values):
        ax2.text(val + 0.01, bar.get_y() + bar.get_height()/2,
                 f'{val:.2f}', va='center', fontsize=11)
    ax2.set_xlim(0, 1.1)
    ax2.set_xlabel('Mean Faithfulness')
    ax2.set_title('Faithfulness by Document Type')
    ax2.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('faithfulness_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: faithfulness_distribution.png')

## 2. Retrieval Ablation Chart

In [ ]:
if not ablation:
    print('No ablation data. Run: python evaluate.py --ablation')
else:
    strategies = list(ablation.keys())
    metrics    = list(ablation[strategies[0]].keys())
    x          = np.arange(len(metrics))
    width      = 0.25
    colors_abl = ['#B4B2A9', '#85B7EB', '#1D9E75']

    fig, ax = plt.subplots(figsize=(9, 5))
    for i, (strategy, color) in enumerate(zip(strategies, colors_abl)):
        vals = [ablation[strategy][m] for m in metrics]
        offset = (i - 1) * width
        bars = ax.bar(x + offset, vals, width, label=strategy.capitalize(),
                      color=color, edgecolor='white')
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                    f'{val:.3f}', ha='center', va='bottom', fontsize=9)

    ax.set_xticks(x)
    ax.set_xticklabels(metrics)
    ax.set_ylim(0, 1.1)
    ax.set_ylabel('Score')
    ax.set_title('Retrieval Ablation: Dense vs Sparse vs Hybrid')
    ax.legend()
    ax.spines[['top','right']].set_visible(False)
    plt.tight_layout()
    plt.savefig('ablation_chart.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: ablation_chart.png')

## 3. Hallucination Examples — Caught by the Detector

In [ ]:
# Show the 5 queries with the lowest faithfulness (most hallucinated)
flagged = per_query[per_query['is_flagged'] == True].sort_values('faithfulness')
print(f'Flagged queries: {len(flagged)} / {len(per_query)}')
print()

for _, row in flagged.head(5).iterrows():
    print(f'Score: {row["faithfulness"]:.2f} | Type: {row["doc_type"]}')
    print(f'Q: {row["question"]}')
    print(f'A (preview): {row["answer_preview"]}')
    print(f'Claims: {row["n_grounded"]} grounded / {row["n_claims"]} total')
    print('-' * 60)

## 4. Summary Table (copy to README)

In [ ]:
print('| Metric | Score |')
print('|--------|-------|')
for k, v in summary.items():
    print(f'| {k.replace("_", " ").title()} | {v:.3f} |')

if ablation:
    print()
    print('| Strategy | ' + ' | '.join(metrics) + ' |')
    print('|----------|' + '|'.join(['------']*len(metrics)) + '|')
    for strategy, vals in ablation.items():
        row_vals = [f'{vals[m]:.3f}' for m in metrics]
        print(f'| {strategy.capitalize()} | ' + ' | '.join(row_vals) + ' |')